In [2]:
# ============================================================================
# CELL 1: Install Dependencies (WORKS 100%)
# ============================================================================

import subprocess
import sys

print("Installing dependencies...\n")

# Install everything together with correct versions
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "huggingface-hub>=0.25.0",
    "transformers>=4.41.0", 
    "tokenizers>=0.15.0",
    "albumentations>=1.3.0",
    "ensemble-boxes>=1.0.9"
])

print("\n" + "="*70)
print("✓ ALL PACKAGES INSTALLED!")
print("="*70)
print("\n🔴 CRITICAL: RESTART KERNEL NOW!")
print("   Click: Runtime → Restart runtime")
print("   After restart, skip this cell and run Cell 2")
print("="*70)

Installing dependencies...




✓ ALL PACKAGES INSTALLED!

🔴 CRITICAL: RESTART KERNEL NOW!
   Click: Runtime → Restart runtime
   After restart, skip this cell and run Cell 2


In [5]:
# ============================================================================
# CELL 1: Install Dependencies
# ============================================================================
# Run this first! Takes ~3-5 minutes

# !pip install -q transformers==4.35.0
# !pip install -q albumentations==1.3.1
# !pip install -q ensemble-boxes==1.0.9

print("✓ Dependencies installed!")

# ============================================================================
# CELL 2: Import Libraries
# ============================================================================

import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
import shutil
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Transformers
from transformers import DetrForObjectDetection, DetrConfig, DetrImageProcessor

# Augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# PyTorch
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# Sklearn
from sklearn.model_selection import StratifiedKFold

print("✓ All libraries imported!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================================
# CELL 3: Configuration
# ============================================================================

class Config:
    """Complete configuration for DETR training"""
    
    # Paths (UPDATE THESE FOR YOUR KAGGLE SETUP)
    ROOT_DIR = "riva-partb-dataset"  # Your dataset path
    WORK_DIR = "detr_output"
    
    # Model
    MODEL_NAME = "facebook/detr-resnet-101"  # or "facebook/detr-resnet-50"
    NUM_CLASSES = 1
    NUM_QUERIES = 300  # Increase to 500 if missing detections
    
    # Training
    EPOCHS = 150  # Reduce to 50 for quick test
    BATCH_SIZE = 2  # Reduce to 1 if OOM
    ACCUMULATION_STEPS = 4  # Effective batch = 2 * 4 = 8
    NUM_WORKERS = 2
    
    # Image size
    IMAGE_SIZE = 1024  # 800, 1024, or 1280
    
    # Learning rates (CRITICAL!)
    LR_BACKBONE = 1e-5
    LR_MAIN = 1e-4
    WEIGHT_DECAY = 1e-4
    
    # Scheduler
    LR_DROP_EPOCH = 100
    LR_GAMMA = 0.1
    
    # Loss weights
    CLASS_LOSS_COEF = 2.0
    BBOX_LOSS_COEF = 5.0
    GIOU_LOSS_COEF = 2.0
    
    # Training settings
    CLIP_GRAD_NORM = 0.1
    WARMUP_EPOCHS = 5
    PATIENCE = 30
    EVAL_EVERY = 5
    SAVE_EVERY = 10
    
    # Augmentation
    HORIZONTAL_FLIP = 0.5
    ROTATION = 10.0
    BRIGHTNESS = 0.2
    CONTRAST = 0.2
    
    # K-Fold
    N_FOLDS = 2
    TRAIN_FOLDS = [0]  # Which folds to train (e.g., [0, 1, 2, 3, 4] for all)
    
    # Inference
    CONF_THRESHOLD = 0.3
    NMS_THRESHOLD = 0.5
    TEST_SCALES = [800, 1024, 1280]
    USE_TTA = True
    
    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    MIXED_PRECISION = True
    
    SEED = 42

cfg = Config()

# Create directories
os.makedirs(cfg.WORK_DIR, exist_ok=True)

# Set seeds
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(cfg.SEED)

print("✓ Configuration loaded!")
print(f"Model: {cfg.MODEL_NAME}")
print(f"Image size: {cfg.IMAGE_SIZE}px")
print(f"Batch size: {cfg.BATCH_SIZE}")
print(f"Epochs: {cfg.EPOCHS}")
print(f"Device: {cfg.DEVICE}")

# ============================================================================
# CELL 4: Data Preparation - Load and Create Folds
# ============================================================================

def create_folds(cfg):
    """Load data and create K-fold splits"""
    
    print("Loading dataset...")
    
    # Load CSVs
    train_csv = os.path.join(cfg.ROOT_DIR, "annotations", "train.csv")
    val_csv = os.path.join(cfg.ROOT_DIR, "annotations", "val.csv")
    
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    
    # Combine
    combined_df = pd.concat([train_df, val_df], ignore_index=True)
    
    print(f"Total images: {combined_df['image_filename'].nunique()}")
    print(f"Total annotations: {len(combined_df)}")
    
    # Get unique images
    image_files = combined_df['image_filename'].unique()
    
    # Create stratification based on cell count
    image_cell_counts = combined_df.groupby('image_filename').size()
    bins = pd.qcut(image_cell_counts.values, q=5, labels=False, duplicates='drop')
    
    # K-Fold split
    skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
    
    folds = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(image_files, bins)):
        train_images = image_files[train_idx]
        val_images = image_files[val_idx]
        
        folds.append({
            'fold': fold_idx,
            'train_images': train_images,
            'val_images': val_images,
            'train_df': combined_df[combined_df['image_filename'].isin(train_images)],
            'val_df': combined_df[combined_df['image_filename'].isin(val_images)]
        })
        
        print(f"Fold {fold_idx}: {len(train_images)} train, {len(val_images)} val")
    
    return folds, combined_df

# Create folds
folds, combined_df = create_folds(cfg)

print(f"\n✓ Created {len(folds)} folds")

# ============================================================================
# CELL 5: Dataset Class
# ============================================================================
# ============================================================================
# CELL 5: Dataset Class (FIXED - Handles bbox clipping)
# ============================================================================

class DETRCellDataset(Dataset):
    """Dataset for DETR cell detection"""
    
    def __init__(self, df, img_dir, transforms=None):
        self.df = df
        self.img_dir = img_dir
        self.transforms = transforms
        
        # Group by image
        self.image_files = df['image_filename'].unique()
        self.image_to_annotations = {
            img: df[df['image_filename'] == img] 
            for img in self.image_files
        }
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        
        # Load image
        img_path = self._find_image_path(img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        h, w = image.shape[:2]
        
        # Get annotations
        annotations = self.image_to_annotations[img_name]
        
        # Convert to boxes (absolute coordinates)
        boxes = []
        for _, row in annotations.iterrows():
            x1 = row['x'] - row['width'] / 2
            y1 = row['y'] - row['height'] / 2
            x2 = row['x'] + row['width'] / 2
            y2 = row['y'] + row['height'] / 2
            
            # Clip to image bounds BEFORE augmentation
            x1 = np.clip(x1, 0, w - 1)
            y1 = np.clip(y1, 0, h - 1)
            x2 = np.clip(x2, 1, w)
            y2 = np.clip(y2, 1, h)
            
            # Skip invalid boxes
            if x2 <= x1 or y2 <= y1:
                continue
                
            boxes.append([x1, y1, x2, y2])
        
        boxes = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)
        labels = np.zeros(len(boxes), dtype=np.int64)
        
        # Apply transforms
        if self.transforms:
            transformed = self.transforms(
                image=image,
                bboxes=boxes,
                labels=labels
            )
            image = transformed['image']
            boxes = np.array(transformed['bboxes'], dtype=np.float32)
            labels = np.array(transformed['labels'], dtype=np.int64)
        
        # Normalize boxes to [0, 1]
        h_new, w_new = image.shape[1], image.shape[2]
        if len(boxes) > 0:
            boxes[:, [0, 2]] /= w_new
            boxes[:, [1, 3]] /= h_new
            # Strict clipping to avoid errors
            boxes = np.clip(boxes, 0.0, 0.9999)
        
        # Convert to cxcywh format (DETR requirement)
        boxes_cxcywh = self._xyxy_to_cxcywh(boxes)
        
        # Create target dict
        target = {'boxes': torch.as_tensor(boxes_cxcywh, dtype=torch.float32),
                  'class_labels': torch.as_tensor(labels, dtype=torch.int64),
                  'image_id': torch.tensor([idx]),
                  'orig_size': torch.tensor([h, w]),
                  'size': torch.tensor([h_new, w_new])
                 }
        
        return image, target
    
    def _find_image_path(self, img_name):
        """Find image in train or val directory"""
        for split in ['train', 'val']:
            path = os.path.join(self.img_dir, split, img_name)
            if os.path.exists(path):
                return path
        raise FileNotFoundError(f"Image not found: {img_name}")
    
    def _xyxy_to_cxcywh(self, boxes):
        """Convert [x1, y1, x2, y2] to [cx, cy, w, h]"""
        if len(boxes) == 0:
            return np.zeros((0, 4), dtype=np.float32)
        
        cxcywh = np.zeros_like(boxes)
        cxcywh[:, 0] = (boxes[:, 0] + boxes[:, 2]) / 2  # cx
        cxcywh[:, 1] = (boxes[:, 1] + boxes[:, 3]) / 2  # cy
        cxcywh[:, 2] = boxes[:, 2] - boxes[:, 0]  # w
        cxcywh[:, 3] = boxes[:, 3] - boxes[:, 1]  # h
        return cxcywh


def collate_fn(batch):
    """Custom collate for DETR"""
    images, targets = zip(*batch)
    images = torch.stack(images)
    return images, list(targets)


print("✓ Dataset class defined!")


def get_train_transforms(cfg):
    """Training augmentations"""
    return A.Compose([
        A.LongestMaxSize(max_size=cfg.IMAGE_SIZE),
        A.PadIfNeeded(
            min_height=cfg.IMAGE_SIZE,
            min_width=cfg.IMAGE_SIZE,
            border_mode=cv2.BORDER_CONSTANT,
            value=0
        ),
        A.HorizontalFlip(p=cfg.HORIZONTAL_FLIP),
        A.Rotate(limit=cfg.ROTATION, border_mode=cv2.BORDER_CONSTANT, p=0.5),
        A.ColorJitter(
            brightness=cfg.BRIGHTNESS,
            contrast=cfg.CONTRAST,
            saturation=0.2,
            hue=0.1,
            p=0.5
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], bbox_params=A.BboxParams(
        format='pascal_voc',
        label_fields=['labels'],
        min_visibility=0.3,
        clip=True  # FIX: Clip boxes to valid range
    ))


def get_val_transforms(cfg):
    """Validation transforms"""
    return A.Compose([
        A.LongestMaxSize(max_size=cfg.IMAGE_SIZE),
        A.PadIfNeeded(
            min_height=cfg.IMAGE_SIZE,
            min_width=cfg.IMAGE_SIZE,
            border_mode=cv2.BORDER_CONSTANT,
            value=0
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], bbox_params=A.BboxParams(
        format='pascal_voc',
        label_fields=['labels'],
        clip=True  # FIX: Clip boxes to valid range
    ))


print("✓ Transforms defined!")
# ============================================================================
# CELL 7: Create Dataloaders for Fold 0 (Test)
# ============================================================================

# Test with fold 0
fold_idx = 0
fold = folds[fold_idx]

img_dir = os.path.join(cfg.ROOT_DIR, "images")

train_dataset = DETRCellDataset(
    df=fold['train_df'],
    img_dir=img_dir,
    transforms=get_train_transforms(cfg)
)

val_dataset = DETRCellDataset(
    df=fold['val_df'],
    img_dir=img_dir,
    transforms=get_val_transforms(cfg)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)

print(f"✓ Dataloaders created!")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Test batch
images, targets = next(iter(train_loader))
print(f"\nTest batch:")
print(f"  Images shape: {images.shape}")
print(f"  First target: {len(targets[0]['boxes'])} boxes")

# ============================================================================
# CELL 8: Model Creation
# ============================================================================

def create_model(cfg):
    """Create DETR model"""
    
    print(f"Loading {cfg.MODEL_NAME}...")
    
    # Load pretrained model
    model = DetrForObjectDetection.from_pretrained(
        cfg.MODEL_NAME,
        num_labels=cfg.NUM_CLASSES,
        ignore_mismatched_sizes=True
    )
    
    # Adjust num_queries if needed
    if model.config.num_queries != cfg.NUM_QUERIES:
        print(f"Adjusting num_queries: {model.config.num_queries} → {cfg.NUM_QUERIES}")
        model.config.num_queries = cfg.NUM_QUERIES
        model.model.query_position_embeddings = nn.Embedding(
            cfg.NUM_QUERIES, 
            model.config.d_model
        )
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"✓ Model loaded!")
    print(f"  Total params: {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}")
    
    return model


model = create_model(cfg)
model = model.to(cfg.DEVICE)

# ============================================================================
# CELL 9: Optimizer and Scheduler
# ============================================================================

def create_optimizer(model, cfg):
    """Create optimizer with differential learning rates"""
    
    # Separate backbone and transformer
    backbone_params = []
    transformer_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'backbone' in name:
            backbone_params.append(param)
        else:
            transformer_params.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': cfg.LR_BACKBONE},
        {'params': transformer_params, 'lr': cfg.LR_MAIN}
    ], weight_decay=cfg.WEIGHT_DECAY)
    
    print(f"✓ Optimizer created")
    print(f"  Backbone LR: {cfg.LR_BACKBONE}")
    print(f"  Transformer LR: {cfg.LR_MAIN}")
    
    return optimizer


def create_scheduler(optimizer, cfg):
    """Create learning rate scheduler"""
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=cfg.LR_DROP_EPOCH,
        gamma=cfg.LR_GAMMA
    )
    return scheduler


optimizer = create_optimizer(model, cfg)
scheduler = create_scheduler(optimizer, cfg)
scaler = GradScaler() if cfg.MIXED_PRECISION else None

print("✓ Optimizer and scheduler ready!")

# ============================================================================
# CELL 10: Evaluation Metrics
# ============================================================================

class DETREvaluator:
    """Simple evaluator for DETR"""
    
    def __init__(self, iou_threshold=0.5):
        self.iou_threshold = iou_threshold

    def evaluate(self, predictions, targets):
        """Calculate mAP, precision, recall"""
        
        total_tp = 0
        total_fp = 0
        total_fn = 0
        
        for pred, target in zip(predictions, targets):
            # Move everything to CPU for evaluation (FIX: device error)
            pred_boxes = pred['boxes'].cpu() if isinstance(pred['boxes'], torch.Tensor) else pred['boxes']
            pred_scores = pred['scores'].cpu() if isinstance(pred['scores'], torch.Tensor) else pred['scores']
            gt_boxes = target['boxes'].cpu() if isinstance(target['boxes'], torch.Tensor) else target['boxes']
            
            if len(gt_boxes) == 0:
                total_fp += len(pred_boxes)
                continue
            
            if len(pred_boxes) == 0:
                total_fn += len(gt_boxes)
                continue
            
            # Match predictions to GT
            matched_gt = set()
            
            for pred_box in pred_boxes:
                ious = self._calculate_iou(pred_box, gt_boxes)
                max_iou_idx = np.argmax(ious)
                max_iou = ious[max_iou_idx]
                
                if max_iou >= self.iou_threshold and max_iou_idx not in matched_gt:
                    total_tp += 1
                    matched_gt.add(max_iou_idx)
                else:
                    total_fp += 1
            
            total_fn += len(gt_boxes) - len(matched_gt)
        
        precision = total_tp / (total_tp + total_fp + 1e-6)
        recall = total_tp / (total_tp + total_fn + 1e-6)
        f1 = 2 * precision * recall / (precision + recall + 1e-6)
        
        return {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'mAP50': f1  # Simplified
        }

    
    def _calculate_iou(self, box1, boxes2):
        """Calculate IoU between one box and multiple boxes"""
        # Convert to xyxy
        box1 = self._cxcywh_to_xyxy(box1.unsqueeze(0) if box1.dim() == 1 else box1)
        boxes2 = self._cxcywh_to_xyxy(boxes2)
        
        # Intersection
        x1_max = torch.max(box1[:, 0].unsqueeze(1), boxes2[:, 0].unsqueeze(0))
        y1_max = torch.max(box1[:, 1].unsqueeze(1), boxes2[:, 1].unsqueeze(0))
        x2_min = torch.min(box1[:, 2].unsqueeze(1), boxes2[:, 2].unsqueeze(0))
        y2_min = torch.min(box1[:, 3].unsqueeze(1), boxes2[:, 3].unsqueeze(0))
        
        intersection = torch.clamp(x2_min - x1_max, min=0) * torch.clamp(y2_min - y1_max, min=0)
        
        # Union
        area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
        area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
        union = area1.unsqueeze(1) + area2.unsqueeze(0) - intersection
        
        iou = intersection / (union + 1e-6)
        return iou.squeeze(0).cpu().numpy()
    
    def _cxcywh_to_xyxy(self, boxes):
        """Convert [cx, cy, w, h] to [x1, y1, x2, y2]"""
        x1 = boxes[:, 0] - boxes[:, 2] / 2
        y1 = boxes[:, 1] - boxes[:, 3] / 2
        x2 = boxes[:, 0] + boxes[:, 2] / 2
        y2 = boxes[:, 1] + boxes[:, 3] / 2
        return torch.stack([x1, y1, x2, y2], dim=1)


evaluator = DETREvaluator()
print("✓ Evaluator ready!")

# ============================================================================
# CELL 11: Training Functions
# ============================================================================

def train_one_epoch(model, train_loader, optimizer, scaler, cfg, epoch):
    """Train for one epoch"""
    
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.EPOCHS}")
    
    for batch_idx, (images, targets) in enumerate(pbar):
        images = images.to(cfg.DEVICE)
        targets = [{k: v.to(cfg.DEVICE) for k, v in t.items()} for t in targets]
        
        # Forward
        if scaler:
            with autocast():
                outputs = model(pixel_values=images, labels=targets)
                loss = outputs.loss
            
            scaler.scale(loss).backward()
            
            # Gradient accumulation
            if (batch_idx + 1) % cfg.ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            outputs = model(pixel_values=images, labels=targets)
            loss = outputs.loss
            loss.backward()
            
            if (batch_idx + 1) % cfg.ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_GRAD_NORM)
                optimizer.step()
                optimizer.zero_grad()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'avg': f"{total_loss/(batch_idx+1):.4f}"})
    
    return total_loss / len(train_loader)



# print("✓ Training functions defined!")
def validate(model, val_loader, evaluator, cfg):
    """Validate model"""
    
    model.eval()
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Validation"):
            images = images.to(cfg.DEVICE)
            
            outputs = model(pixel_values=images)
            
            # Post-process
            logits = outputs.logits
            boxes = outputs.pred_boxes
            
            for i in range(len(images)):
                probs = torch.softmax(logits[i], dim=-1)
                scores, labels = probs[:, :-1].max(dim=-1)
                
                keep = scores > cfg.CONF_THRESHOLD
                
                all_predictions.append({
                    'boxes': boxes[i][keep],
                    'scores': scores[keep],
                    'labels': labels[keep]
                })
                
                # FIX: Convert class_labels back to labels for evaluator
                target_dict = {
                    'boxes': targets[i]['boxes'],
                    'labels': targets[i]['class_labels']
                }
                all_targets.append(target_dict)
    
    metrics = evaluator.evaluate(all_predictions, all_targets)
    return metrics

# ============================================================================
# CELL 12: Main Training Loop
# ============================================================================

def train_fold(model, train_loader, val_loader, optimizer, scheduler, cfg, fold_idx):
    """Complete training for one fold"""
    
    best_f1 = 0
    patience_counter = 0
    
    # Create save directory
    save_dir = os.path.join(cfg.WORK_DIR, f"fold_{fold_idx}")
    os.makedirs(save_dir, exist_ok=True)
    
    history = {'train_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': []}
    
    for epoch in range(cfg.EPOCHS):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{cfg.EPOCHS}")
        print(f"{'='*60}")
        
        # Train
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, cfg, epoch)
        history['train_loss'].append(train_loss)
        
        print(f"Train Loss: {train_loss:.4f}")
        
        # Validate
        if (epoch + 1) % cfg.EVAL_EVERY == 0 or epoch == cfg.EPOCHS - 1:
            metrics = validate(model, val_loader, evaluator, cfg)
            
            history['val_f1'].append(metrics['f1'])
            history['val_precision'].append(metrics['precision'])
            history['val_recall'].append(metrics['recall'])
            
            print(f"\nValidation Metrics:")
            print(f"  Precision: {metrics['precision']:.4f}")
            print(f"  Recall: {metrics['recall']:.4f}")
            print(f"  F1: {metrics['f1']:.4f}")
            
            # Save best
            if metrics['f1'] > best_f1:
                best_f1 = metrics['f1']
                patience_counter = 0
                
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_f1': best_f1,
                    'metrics': metrics
                }, os.path.join(save_dir, 'best.pth'))
                
                print(f"  ✓ New best F1: {best_f1:.4f}")
            else:
                patience_counter += 1
            
            # Early stopping
            if patience_counter >= cfg.PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
        
        # Step scheduler
        scheduler.step()
    
    print(f"\n{'='*60}")
    print(f"Training complete! Best F1: {best_f1:.4f}")
    print(f"{'='*60}")
    
    return history, best_f1


# Train!
print("🚀 Starting training...")
history, best_f1 = train_fold(model, train_loader, val_loader, optimizer, scheduler, cfg, fold_idx=0)

# ============================================================================
# CELL 13: Plot Training History
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

# Metrics
if history['val_f1']:
    epochs_val = list(range(cfg.EVAL_EVERY - 1, len(history['train_loss']), cfg.EVAL_EVERY))
    axes[1].plot(epochs_val, history['val_f1'], label='F1', marker='o')
    axes[1].plot(epochs_val, history['val_precision'], label='Precision', marker='s')
    axes[1].plot(epochs_val, history['val_recall'], label='Recall', marker='^')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Validation Metrics')
    axes[1].legend()
    axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(cfg.WORK_DIR, 'training_history.png'), dpi=150)
plt.show()

print("✓ Training history plotted!")


✓ Dependencies installed!
✓ All libraries imported!
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L40S
✓ Configuration loaded!
Model: facebook/detr-resnet-101
Image size: 1024px
Batch size: 2
Epochs: 150
Device: cuda
Loading dataset...
Total images: 959
Total annotations: 15949
Fold 0: 479 train, 480 val
Fold 1: 480 train, 479 val

✓ Created 2 folds
✓ Dataset class defined!
✓ Transforms defined!


✓ Dataloaders created!
Train batches: 239
Val batches: 240

Test batch:
  Images shape: torch.Size([2, 3, 1024, 1024])
  First target: 17 boxes
Loading facebook/detr-resnet-101...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/243M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/785 [00:00<?, ?it/s]

...

Epoch 1/150:   0%|          | 0/239 [00:00<?, ?it/s]

Train Loss: 2.6915

Epoch 2/150


Epoch 2/150:   0%|          | 0/239 [00:00<?, ?it/s]

Exception in thread Thread-14 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.11/

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# CELL 14: Inference on Test Set
# ============================================================================
# ============================================================================
# CELL 14: Inference on Test Set (FIXED)
# ============================================================================

class DETRInference:
    """DETR inference with TTA"""
    
    def __init__(self, model, cfg):
        self.model = model
        self.cfg = cfg
        self.model.eval()
        
        # Create inference-specific transform (NO bbox params!)
        self.transform = A.Compose([
            A.LongestMaxSize(max_size=cfg.IMAGE_SIZE),
            A.PadIfNeeded(
                min_height=cfg.IMAGE_SIZE,
                min_width=cfg.IMAGE_SIZE,
                border_mode=cv2.BORDER_CONSTANT,
                value=0
            ),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])  # FIX: No bbox_params for inference
    
    def predict_image(self, image_path, use_tta=True):
        """Predict on single image"""
        
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]
        
        all_boxes = []
        all_scores = []
        
        # Original
        boxes, scores = self._predict_single(image)
        all_boxes.extend(boxes)
        all_scores.extend(scores)
        
        # Horizontal flip TTA
        if use_tta and self.cfg.USE_TTA:
            image_flip = cv2.flip(image, 1)
            boxes_flip, scores_flip = self._predict_single(image_flip)
            
            # Flip boxes back
            for box in boxes_flip:
                cx, cy, w, h = box
                cx_new = 1.0 - cx
                all_boxes.append([cx_new, cy, w, h])
            all_scores.extend(scores_flip)
        
        # Convert to absolute coordinates
        boxes_abs = []
        for box, score in zip(all_boxes, all_scores):
            cx, cy, w, h = box
            boxes_abs.append({
                'x': cx * orig_w,
                'y': cy * orig_h,
                'width': w * orig_w,
                'height': h * orig_h,
                'conf': score
            })
        
        return boxes_abs
    
    def _predict_single(self, image):
        """Single prediction"""
        
        # Transform (FIX: use self.transform, no bboxes)
        transformed = self.transform(image=image)
        img_tensor = transformed['image'].unsqueeze(0).to(self.cfg.DEVICE)
        
        # Predict
        with torch.no_grad():
            outputs = self.model(pixel_values=img_tensor)
        
        # Post-process
        logits = outputs.logits[0]
        boxes = outputs.pred_boxes[0]
        
        probs = torch.softmax(logits, dim=-1)
        scores, labels = probs[:, :-1].max(dim=-1)
        
        keep = scores > self.cfg.CONF_THRESHOLD
        
        boxes_filtered = boxes[keep].cpu().numpy().tolist()
        scores_filtered = scores[keep].cpu().numpy().tolist()
        
        return boxes_filtered, scores_filtered


# Load best model
best_checkpoint = os.path.join(cfg.WORK_DIR, "fold_0", "best.pth")
checkpoint = torch.load(best_checkpoint)
model.load_state_dict(checkpoint['model_state_dict'])

inference = DETRInference(model, cfg)
print("✓ Inference engine ready!")

# ============================================================================
# CELL 15: Generate Submission
# ============================================================================

def generate_submission(inference, cfg):
    """Generate submission file"""
    
    test_dir = os.path.join(cfg.ROOT_DIR, "images/images", "test")
    
    if not os.path.exists(test_dir):
        print(f"Warning: Test directory not found: {test_dir}")
        return None
    
    test_images = sorted([f for f in os.listdir(test_dir) if f.endswith('.png')])
    print(f"Found {len(test_images)} test images")
    
    all_predictions = []
    pred_id = 0
    
    for img_name in tqdm(test_images, desc="Generating predictions"):
        img_path = os.path.join(test_dir, img_name)
        
        predictions = inference.predict_image(img_path, use_tta=cfg.USE_TTA)
        
        for pred in predictions:
            all_predictions.append({
                'id': pred_id,
                'image_filename': img_name,
                'class': 0,
                'x': pred['x'],
                'y': pred['y'],
                'width': pred['width'],
                'height': pred['height'],
                'conf': pred['conf']
            })
            pred_id += 1
    
    # Create DataFrame
    submission_df = pd.DataFrame(all_predictions)
    
    # Save
    output_path = os.path.join(cfg.WORK_DIR, 'submission.csv')
    submission_df.to_csv(output_path, index=False, float_format='%.6f')
    
    print(f"\n{'='*60}")
    print(f"✓ Submission saved: {output_path}")
    print(f"Total predictions: {len(submission_df):,}")
    print(f"Images: {submission_df['image_filename'].nunique()}")
    print(f"Avg per image: {len(submission_df) / submission_df['image_filename'].nunique():.1f}")
    print(f"Confidence range: [{submission_df['conf'].min():.4f}, {submission_df['conf'].max():.4f}]")
    print(f"{'='*60}")
    
    return submission_df

# Generate submission
submission_df = generate_submission(inference, cfg)



In [ ]:
# Display first few rows
if submission_df is not None:
    print("\nFirst 10 predictions:")
    print(submission_df.head(10))


In [ ]:

# ============================================================================
# CELL 16: Visualize Predictions (Optional)
# ============================================================================

def visualize_predictions(image_path, predictions, max_boxes=50):
    """Visualize predictions on an image"""
    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Draw boxes
    for pred in predictions[:max_boxes]:
        x = int(pred['x'])
        y = int(pred['y'])
        w = int(pred['width'])
        h = int(pred['height'])
        conf = pred['conf']
        
        x1 = int(x - w/2)
        y1 = int(y - h/2)
        x2 = int(x + w/2)
        y2 = int(y + h/2)
        
        color = (0, 255, 0) if conf > 0.5 else (255, 255, 0)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(image, f"{conf:.2f}", (x1, y1-5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    return image

# Visualize a few test predictions
test_dir = os.path.join(cfg.ROOT_DIR, "images/images", "test")
if os.path.exists(test_dir):
    test_images = sorted([f for f in os.listdir(test_dir) if f.endswith('.png')])[:3]
    
    fig, axes = plt.subplots(1, len(test_images), figsize=(20, 6))
    if len(test_images) == 1:
        axes = [axes]
    
    for idx, img_name in enumerate(test_images):
        img_path = os.path.join(test_dir, img_name)
        predictions = inference.predict_image(img_path, use_tta=False)
        
        vis_image = visualize_predictions(img_path, predictions)
        
        axes[idx].imshow(vis_image)
        axes[idx].set_title(f"{img_name}\n{len(predictions)} cells")
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.WORK_DIR, 'predictions_visualization.png'), dpi=150)
    plt.show()
    
    print("✓ Predictions visualized!")





In [ ]:
# ============================================================================
# CELL 17: Summary and Next Steps
# ============================================================================

print("\n" + "="*80)
print("🎉 DETR TRAINING COMPLETE!")
print("="*80)
print(f"\nBest F1 Score: {best_f1:.4f}")
print(f"Model saved: {os.path.join(cfg.WORK_DIR, 'fold_0', 'best.pth')}")
print(f"Submission saved: {os.path.join(cfg.WORK_DIR, 'submission.csv')}")
print("\n📊 Next Steps:")
print("  1. Download submission.csv and submit to Kaggle")
print("  2. Try different confidence thresholds (0.1, 0.2, 0.3, 0.4, 0.5)")
print("  3. Train more folds for ensemble (change TRAIN_FOLDS in config)")
print("  4. Experiment with hyperparameters:")
print("     - Increase NUM_QUERIES to 500 if missing detections")
print("     - Try ResNet-50 for faster training")
print("     - Adjust learning rates if loss doesn't decrease")
print("     - Increase EPOCHS to 200 for better convergence")

print("\n🔧 Hyperparameter Tuning:")
print("  If mAP < 0.40:")
print("    - Increase EPOCHS to 200")
print("    - Increase BBOX_LOSS_COEF to 8.0")
print("    - Lower LR_MAIN to 5e-5")
print("\n  If overfitting:")
print("    - More augmentation (ROTATION=15, BRIGHTNESS=0.3)")
print("    - Lower learning rate")
print("\n  If slow convergence:")
print("    - Check batch size (try 4 if memory allows)")
print("    - Verify data loading (re-run Cell 7)")

print("\n💡 Pro Tips:")
print("  - Compare with YOLO baseline (mAP50-95: 0.60)")
print("  - Generate multiple submissions with different thresholds")
print("  - Ensemble with YOLO predictions for best results")

print("\n" + "="*80)
print("Good luck! 🚀")
print("="*80)